In [1]:
import pandas as pd
import numpy as np
import json
import torch
import torch.nn as nn
import itertools
from pathlib import Path
from datetime import datetime
from torch.optim.lr_scheduler import CyclicLR
from torch.utils.data import Dataset, DataLoader
import mlflow
import mlflow.pytorch
from mlflow.tracking import MlflowClient
from glob import glob
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support
import os
import warnings
warnings.filterwarnings('ignore')

In [11]:
PROJECT_ROOT = Path("D:/GitRepo/student-performance-forecast")

# Директории для MLflow
MLFLOW_DIR = PROJECT_ROOT / "mlflow"
MLFLOW_DIR.mkdir(exist_ok=True)

# Директория для артефактов
ARTIFACTS_DIR = MLFLOW_DIR / "artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)

# Директория для временных файлов (model_stats.json и др.)
TEMP_DIR = PROJECT_ROOT / "temp"
TEMP_DIR.mkdir(exist_ok=True)

# Устанавливаем tracking URI
mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DIR / 'mlflow.db'}")

In [3]:
class DKTSequenceDataset(Dataset):    
    def __init__(self, sequences, feature_cols, target_col, max_len=50):
        self.sequences = sequences
        self.feature_cols = feature_cols
        self.target_col = target_col
        self.max_len = max_len
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        df = self.sequences[idx]        
        X = df[self.feature_cols].values
        y = df[self.target_col].values
        if len(X) > self.max_len:
            X = X[-self.max_len:]
            y = y[-self.max_len:]  
            
        seq_len = len(X)

        X_pad = np.zeros((self.max_len, len(self.feature_cols)))
        y_pad = np.zeros(self.max_len)
        mask = np.zeros(self.max_len)
        
        X_pad[:seq_len] = X
        y_pad[:seq_len] = y
        mask[:seq_len] = 1
        
        return {
            'X': torch.FloatTensor(X_pad),
            'y': torch.FloatTensor(y_pad),
            'mask': torch.FloatTensor(mask),
            'seq_len': seq_len
        }

In [4]:
class SimpleDKT(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, n_heads=4, dropout=0.1):
        super().__init__()
        
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.pos_embedding = nn.Embedding(1000, hidden_dim)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=n_heads,
            dim_feedforward=hidden_dim * 4,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.output = nn.Linear(hidden_dim, 1)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask):
        # x: [batch, seq_len, features]
        # mask: [batch, seq_len]
        
        batch_size, seq_len, _ = x.shape
        x = self.input_proj(x)
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0)
        x = x + self.pos_embedding(positions)
        x = self.dropout(x)
        src_key_padding_mask = (mask == 0)
        x = self.transformer(x, src_key_padding_mask=src_key_padding_mask)
        
        logits = self.output(x).squeeze(-1)
        
        return torch.sigmoid(logits)

In [5]:
def load_sequences(file_pattern, feature_cols, target_col, sample_ratio=1.0):
    files = glob(file_pattern)
    if sample_ratio < 1.0:
        n_files = int(len(files) * sample_ratio)
        files = np.random.choice(files, n_files, replace=False)
        #print(f"Используется: {len(files)} файлов ({sample_ratio*100:.1f}%)")
    
    sequences = []
    for file_path in files:
        if file_path.endswith('.parquet'):
            df = pd.read_parquet(file_path)
        else:
            df = pd.read_csv(file_path)
        available_feats = [col for col in feature_cols if col in df.columns]
        if target_col not in df.columns or not available_feats:
            continue
        df = df[available_feats + [target_col]].copy()
        for col in available_feats:
            if df[col].dtype == 'object':
                df[col] = pd.Categorical(df[col]).codes
        df = df.fillna(0)
        if len(df) >= 10:
            sequences.append(df)
    
    #print(f"Загружено последовательностей: {len(sequences)}")
    if sequences:
        lengths = [len(s) for s in sequences]
        #print(f"Средняя длина: {np.mean(lengths):.1f}, мин: {min(lengths)}, макс: {max(lengths)}")
    
    return sequences


def normalize_sequences(sequences, feature_cols):
    all_values = np.vstack([seq[feature_cols].values for seq in sequences])
    means = all_values.mean(axis=0)
    stds = all_values.std(axis=0)
    stds[stds == 0] = 1
    
    for seq in sequences:
        seq[feature_cols] = (seq[feature_cols].values - means) / stds
    
    return sequences, means, stds

In [6]:
def train_dkt(
    train_loader,
    val_loader,
    features_num,
    epochs=20,
    learning_rate=0.001,
    hidden_dim=64,
    num_layers=2,
    device='cpu'
):
    model = SimpleDKT(
        input_dim=features_num,
        hidden_dim=hidden_dim,
        num_layers=num_layers
    ).to(device)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    scheduler = CyclicLR(optimizer, base_lr=learning_rate, max_lr=0.01, step_size_up=3)
    criterion = nn.BCELoss()
    
    train_losses = []
    train_accs = []
    val_losses = []
    val_accs = []
    
    best_val_loss = float('inf')
    
    for epoch in range(epochs):
        # Train
        model.train()
        train_loss = 0
        train_acc = 0
        train_count = 0
        
        for batch in train_loader:
            X = batch['X'].to(device)
            y = batch['y'].to(device)
            mask = batch['mask'].to(device)
            
            y_binary = (y >= 0.5).float()
            
            pred = model(X, mask)
            pred_next = pred[:, :-1]
            y_next = y_binary[:, 1:]
            mask_next = mask[:, :-1]
            
            loss_mask = mask_next.bool()
            if loss_mask.sum() > 0:
                loss = criterion(pred_next[loss_mask], y_next[loss_mask])
                
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                scheduler.step()
                
                train_loss += loss.item()
                
                pred_class = (pred_next > 0.5).float()
                train_acc += (pred_class[loss_mask] == y_next[loss_mask]).sum().item()
                train_count += loss_mask.sum().item()
        
        avg_train_loss = train_loss / len(train_loader)
        avg_train_acc = train_acc / train_count if train_count > 0 else 0
        
        # Validation
        model.eval()
        val_loss = 0
        val_acc = 0
        val_count = 0
        
        with torch.no_grad():
            for batch in val_loader:
                X = batch['X'].to(device)
                y = batch['y'].to(device)
                mask = batch['mask'].to(device)
                
                y_binary = (y >= 0.5).float()
                
                pred = model(X, mask)
                pred_next = pred[:, :-1]
                y_next = y_binary[:, 1:]
                mask_next = mask[:, :-1]
                
                loss_mask = mask_next.bool()
                if loss_mask.sum() > 0:
                    loss = criterion(pred_next[loss_mask], y_next[loss_mask])
                    val_loss += loss.item()
                    
                    pred_class = (pred_next > 0.5).float()
                    val_acc += (pred_class[loss_mask] == y_next[loss_mask]).sum().item()
                    val_count += loss_mask.sum().item()
        
        avg_val_loss = val_loss / len(val_loader)
        avg_val_acc = val_acc / val_count if val_count > 0 else 0

        train_losses.append(avg_train_loss)
        train_accs.append(avg_train_acc)
        val_losses.append(avg_val_loss)
        val_accs.append(avg_val_acc)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            
    print(f"==== Best Val Loss: {best_val_loss:.4f} ====")
    print(f"==== Best Val Acc: {max(val_accs):.4f} ====")
    
    return model, {
        'train_loss': train_losses,
        'train_acc': train_accs,
        'val_loss': val_losses,
        'val_acc': val_accs
    }

In [12]:
def train_dkt_mlflow(
    file_pattern,
    feature_cols,
    target_col='accuracy',
    sample_ratio=1.0,
    max_seq_len=100,
    batch_size=32,
    epochs=20,
    learning_rate=0.001,
    hidden_dim=128,
    num_layers=4,
    val_split=0.2,
    experiment_name="DKT_EdNET",
    device = 'cpu'
):
    run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    mlflow.set_experiment(experiment_name)

    with mlflow.start_run() as run:
        sequences = load_sequences(file_pattern, feature_cols, target_col, sample_ratio)
        if len(sequences) == 0:
            print("Ошибка: нет данных для обучения!")
            return None, None
        sequences, means, stds = normalize_sequences(sequences, feature_cols)
        train_seqs, val_seqs = train_test_split(sequences, test_size=val_split, random_state=42)
    
        train_dataset = DKTSequenceDataset(train_seqs, feature_cols, target_col, max_seq_len)
        val_dataset = DKTSequenceDataset(val_seqs, feature_cols, target_col, max_seq_len)
        
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

        model, stats = train_dkt(
            train_loader,
            val_loader,
            len(feature_cols),
            epochs=epochs,
            learning_rate=learning_rate,
            hidden_dim=hidden_dim,
            num_layers=num_layers,
            device=device
        )
        stats['means'] = means
        stats['stds'] = stds
        
        mlflow.log_params({
            "sample_ratio": sample_ratio,
            "max_seq_len": max_seq_len,
            "batch_size": batch_size,
            "epochs": epochs,
            "learning_rate": learning_rate,
            "hidden_dim": hidden_dim,
            "num_layers": num_layers,
            "feature_cols": str(feature_cols),
            "target_col": target_col,
            "total_params": sum(p.numel() for p in model.parameters()),
            "trainable_params": sum(p.numel() for p in model.parameters() if p.requires_grad),
            "model_size_mb": sum(p.numel() * p.element_size() for p in model.parameters()) / 1024**2
        })
        
        if stats is not None:
            best_val_acc = max(stats.get('val_acc', [0]))
            best_val_loss = min(stats.get('val_loss', [float('inf')]))
            all_preds, all_targets = collect_predictions(model, val_loader)
            
            if len(all_preds) > 0:
                auc_score = roc_auc_score(all_targets, all_preds)
                precision, recall, f1, _ = precision_recall_fscore_support(
                    all_targets, 
                    (np.array(all_preds) > 0.5).astype(int), 
                    average='binary'
                )
            else:
                auc_score = 0
                precision = recall = f1 = 0
            
            train_losses = stats.get('train_loss', [])
            train_accs = stats.get('train_acc', [])
            val_losses = stats.get('val_loss', [])
            val_accs = stats.get('val_acc', [])
            
            best_epoch = np.argmin(val_losses) if val_losses else 0
            convergence_speed = best_epoch
            overfitting_gap = (val_losses[-1] - train_losses[-1]) if train_losses and val_losses else 0
            
            mlflow.log_metrics({
                "best_val_accuracy": best_val_acc,
                "best_val_loss": best_val_loss,
                "final_train_loss": train_losses[-1] if train_losses else 0,
                "final_train_acc": train_accs[-1] if train_accs else 0,
                "final_val_loss": val_losses[-1] if val_losses else 0,
                "final_val_acc": val_accs[-1] if val_accs else 0,
                "auc_score": auc_score,
                "precision": precision,
                "recall": recall,
                "f1_score": f1,
                "best_epoch": best_epoch,
                "convergence_speed": convergence_speed,
                "overfitting_gap": overfitting_gap,
                "train_loss_std": np.std(train_losses) if len(train_losses) > 1 else 0,
                "val_loss_std": np.std(val_losses) if len(val_losses) > 1 else 0,
                "train_acc_std": np.std(train_accs) if len(train_accs) > 1 else 0,
                "val_acc_std": np.std(val_accs) if len(val_accs) > 1 else 0,
                "loss_improvement": ((val_losses[0] - val_losses[-1]) / val_losses[0] * 100) if val_losses and val_losses[0] > 0 else 0,
                "acc_improvement": ((val_accs[-1] - val_accs[0]) * 100) if val_accs else 0
            })
        
        mlflow.pytorch.log_model(
            model,
            registered_model_name=f"DKT_LR{learning_rate}_HidDim{hidden_dim}_NumL{num_layers}_Batch{batch_size}"
        )
            
        if stats:
            stats_file = TEMP_DIR / f"model_stats_{run_timestamp}.json"
            stats_to_save = {
                "means": stats.get('means', []).tolist() if hasattr(stats.get('means', []), 'tolist') else [],
                "stds": stats.get('stds', []).tolist() if hasattr(stats.get('stds', []), 'tolist') else [],
                "feature_cols": feature_cols,
                "training_history": {
                    "train_loss": stats.get('train_loss', []),
                    "train_acc": stats.get('train_acc', []),
                    "val_loss": stats.get('val_loss', []),
                    "val_acc": stats.get('val_acc', [])
                }
            }
            with open(stats_file, "w") as f:
                json.dump(stats_to_save, f)
            mlflow.log_artifact(str(stats_file))

            stats_file.unlink()
        
        run_id = run.info.run_id
        print(f"MLflow Run ID: {run_id}")
        return model, stats, run_id


def collect_predictions(model, dataloader, device='cuda'):
    model.eval()
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for batch in dataloader:
            X = batch['X'].to(device)
            y = batch['y'].to(device)
            mask = batch['mask'].to(device)
            
            y_binary = (y >= 0.5).float()
            pred = model(X, mask)
            
            pred_next = pred[:, :-1]
            y_next = y_binary[:, 1:]
            mask_next = mask[:, :-1]
            
            loss_mask = mask_next.bool()
            if loss_mask.sum() > 0:
                all_preds.extend(pred_next[loss_mask].cpu().numpy())
                all_targets.extend(y_next[loss_mask].cpu().numpy())
    
    return all_preds, all_targets

In [8]:
def grid_search_dkt(file_pattern, base_params, param_grid):
    keys = param_grid.keys()
    values = param_grid.values()
    combinations = list(itertools.product(*values))
    
    results = []
    
    for combo in combinations:
        params = base_params.copy()
        for key, value in zip(keys, combo):
            params[key] = value
        
        print(f"\n{'='*60}")
        print(f"batch_size: {params.get('batch_size')}, hidden_dim: {params.get('hidden_dim')}, "
      f"num_layers: {params.get('num_layers')}, lr: {params.get('learning_rate')}")
        print('='*60)

        model, stats, run_id = train_dkt_mlflow(
            file_pattern=file_pattern,
            feature_cols=params['feature_cols'],
            target_col=params['target_col'],
            sample_ratio=params.get('sample_ratio', 1.0),
            max_seq_len=params.get('max_seq_len', 100),
            batch_size=params['batch_size'],
            epochs=params['epochs'],
            learning_rate=params.get('learning_rate', 0.001),
            hidden_dim=params['hidden_dim'],
            num_layers=params['num_layers'],
            experiment_name="DKT_GridSearch",
            device = 'cuda' if torch.cuda.is_available() else 'cpu'
        )
        
        best_val_acc = max(stats['val_acc']) if stats['val_acc'] else 0
        best_val_loss = min(stats['val_loss']) if stats['val_loss'] else float('inf')
        
        results.append({
            'params': params,
            'run_id': run_id,
            'best_val_acc': best_val_acc,
            'best_val_loss': best_val_loss
        })
    
    print("\n" + "="*60)
    print("RESULTS")
    print("="*60)
    for res in sorted(results, key=lambda x: x['best_val_acc'], reverse=True):
        print(f"Val Acc: {res['best_val_acc']:.4f} | Params: {res['params']}")
    
    return results

In [9]:
def predict_next(model, student_df, feature_cols, stats, device='cuda'):
    """
    Предсказание вероятности правильного ответа на следующем бандле
    """
    model.eval()
    
    # Нормализация
    X = student_df[feature_cols].values
    X = (X - stats['means']) / stats['stds']
    
    # Добавляем batch dimension: [seq_len, features] -> [1, seq_len, features]
    X = torch.FloatTensor(X).unsqueeze(0)  # [1, seq_len, features]
    
    # Маска должна быть [1, seq_len]
    seq_len = X.shape[1]
    mask = torch.ones(1, seq_len)  # [1, seq_len]
    
    with torch.no_grad():
        pred = model(X.to(device), mask.to(device))
        # pred имеет размер [1, seq_len]
        # Берем последний элемент последовательности
        next_prob = pred[0, -1].item()
    
    return next_prob

In [14]:
param_grid = {
    'hidden_dim': [32, 64, 128],
    'num_layers': [2, 4],
    'batch_size': [128, 256],
    'learning_rate': [0.001, 0.0005]
}

base_params = {
    'file_pattern': "D:/GitRepo/student-performance-forecast/data/processed/EdNET_KT3/*.parquet",
    'feature_cols': ['bundle_time_ms', 'viewed_lectures', 'hour_sin', 'is_weekend', 
                     'viewed_explanations', 'n_questions', 'n_attempts', 'accuracy_ma3'],
    'target_col': 'accuracy',
    'sample_ratio': 0.9,
    'max_seq_len': 138,
    'epochs': 20
}

results = grid_search_dkt(base_params['file_pattern'], base_params, param_grid)


batch_size: 128, hidden_dim: 32, num_layers: 2, lr: 0.001
==== Best Val Loss: 0.2062 ====
==== Best Val Acc: 0.9053 ====


2026/05/13 10:53:53 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
Successfully registered model 'DKT_LR0.001_HidDim32_NumL2_Batch128'.
Created version '1' of model 'DKT_LR0.001_HidDim32_NumL2_Batch128'.


MLflow Run ID: 1bfbe3bcde3f4c36afcb8247488bc806

batch_size: 128, hidden_dim: 32, num_layers: 2, lr: 0.0005
==== Best Val Loss: 0.1840 ====
==== Best Val Acc: 0.9137 ====


2026/05/13 10:57:35 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
Successfully registered model 'DKT_LR0.0005_HidDim32_NumL2_Batch128'.
Created version '1' of model 'DKT_LR0.0005_HidDim32_NumL2_Batch128'.


MLflow Run ID: f891a7be3e784aaf97a1307f100e9ef8

batch_size: 256, hidden_dim: 32, num_layers: 2, lr: 0.001
==== Best Val Loss: 0.2141 ====
==== Best Val Acc: 0.9001 ====


...

MLflow Run ID: 7ec179a510d242bc96b94b071a36c38d

batch_size: 256, hidden_dim: 128, num_layers: 4, lr: 0.001
==== Best Val Loss: 0.6481 ====
==== Best Val Acc: 0.6409 ====


2026/05/13 12:17:40 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
Successfully registered model 'DKT_LR0.001_HidDim128_NumL4_Batch256'.
Created version '1' of model 'DKT_LR0.001_HidDim128_NumL4_Batch256'.


MLflow Run ID: df2930bb34044feeb345a4b5cbade8c3

batch_size: 256, hidden_dim: 128, num_layers: 4, lr: 0.0005
==== Best Val Loss: 0.6541 ====
==== Best Val Acc: 0.6401 ====


2026/05/13 12:21:36 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
Successfully registered model 'DKT_LR0.0005_HidDim128_NumL4_Batch256'.
Created version '1' of model 'DKT_LR0.0005_HidDim128_NumL4_Batch256'.


MLflow Run ID: 02e4b44ba52540c0949650feb1fb0b27

RESULTS
Val Acc: 0.9678 | Params: {'file_pattern': 'D:/GitRepo/student-performance-forecast/data/processed/EdNET_KT3/*.parquet', 'feature_cols': ['bundle_time_ms', 'viewed_lectures', 'hour_sin', 'is_weekend', 'viewed_explanations', 'n_questions', 'n_attempts', 'accuracy_ma3'], 'target_col': 'accuracy', 'sample_ratio': 0.9, 'max_seq_len': 138, 'epochs': 20, 'hidden_dim': 128, 'num_layers': 2, 'batch_size': 128, 'learning_rate': 0.001}
Val Acc: 0.9649 | Params: {'file_pattern': 'D:/GitRepo/student-performance-forecast/data/processed/EdNET_KT3/*.parquet', 'feature_cols': ['bundle_time_ms', 'viewed_lectures', 'hour_sin', 'is_weekend', 'viewed_explanations', 'n_questions', 'n_attempts', 'accuracy_ma3'], 'target_col': 'accuracy', 'sample_ratio': 0.9, 'max_seq_len': 138, 'epochs': 20, 'hidden_dim': 128, 'num_layers': 2, 'batch_size': 256, 'learning_rate': 0.001}
Val Acc: 0.9642 | Params: {'file_pattern': 'D:/GitRepo/student-performance-forecast